### Завдання 1. Завантаження та очищення даних
Завантажити та відкрити датасет "Individual Household Electric Power Consumption Dataset". 
Здійснити data cleaning (видалити або заповнити пропущені значення). 
Усі наступні вибірки необхідно реалізувати окремими функціями та проаналізувати часові витрати на їх виконання за допомогою модуля `timeit`.

In [1]:
import pandas as pd
import urllib.request
import zipfile
import os
import timeit

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
extract_folder = "power_data"

if not os.path.exists(extract_folder):
    print("Завантаження датасету.")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    print("Розпакування завершено.")
else:
    print("Датасет вже завантажено.")

txt_file = os.path.join(extract_folder, "household_power_consumption.txt")
print("Зчитування даних у DataFrame")

df_power = pd.read_csv(txt_file, sep=';', na_values=['?'], low_memory=False)

df_power = df_power.dropna()

print(f"Дані успішно очищено! Залишилося {len(df_power)} записів.")
df_power.head()

Завантаження датасету.
Розпакування завершено.
Зчитування даних у DataFrame
Дані успішно очищено! Залишилося 2049280 записів.


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Завдання 2. Вибірка за загальною активною потужністю
Обрати всі записи, у яких загальна активна споживана потужність (`Global_active_power`) перевищує 5 кВт.

In [2]:
import timeit

def filter_high_power(df):
    return df[df['Global_active_power'] > 5.0]

high_power_df = filter_high_power(df_power)

time_taken = timeit.timeit(lambda: filter_high_power(df_power), number=10) / 10

print(f"Знайдено записів: {len(high_power_df)}")
print(f"Середній час виконання: {time_taken:.5f} секунд")
high_power_df.head()

Знайдено записів: 17547
Середній час виконання: 0.00744 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


### Завдання 3. Вибірка за силою струму та споживанням приладів
Обрати всі записи, у яких сила струму лежить в межах 19-20 А. Для них виявити ті, у яких пральна машина та холодильник (`Sub_metering_2`) споживають більше, ніж бойлер та кондиціонер (`Sub_metering_3`).

In [3]:
def filter_intensity_and_appliances(df):
    return df[(df['Global_intensity'] >= 19) & 
              (df['Global_intensity'] <= 20) & 
              (df['Sub_metering_2'] > df['Sub_metering_3'])]

filtered_appliances_df = filter_intensity_and_appliances(df_power)

time_taken = timeit.timeit(lambda: filter_intensity_and_appliances(df_power), number=10) / 10

print(f"Знайдено записів: {len(filtered_appliances_df)}")
print(f"Середній час виконання: {time_taken:.5f} секунд")
filtered_appliances_df.head()

Знайдено записів: 2509
Середній час виконання: 0.01267 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


### Завдання 4. Випадкова вибірка
Обрати випадковим чином 500 000 записів (без повторів елементів вибірки). Для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [4]:
def sample_and_calculate_means(df):
    sample_df = df.sample(n=500000, replace=False)
    return sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

means_result = sample_and_calculate_means(df_power)

time_taken = timeit.timeit(lambda: sample_and_calculate_means(df_power), number=5) / 5

print("Середні величини груп споживання:")
print(means_result)
print(f"\nСередній час виконання: {time_taken:.5f} секунд")

Середні величини груп споживання:
Sub_metering_1    1.126494
Sub_metering_2    1.304728
Sub_metering_3    6.455784
dtype: float64

Середній час виконання: 0.24546 секунд


### Завдання 5. Комплексна вечірня вибірка
Обрати ті записи, які після 18:00 споживають понад 6 кВт за хвилину в середньому. Серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група `Sub_metering_2` є найбільшою). Після цього обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [5]:
def complex_evening_filter(df):
    res = df[(df['Time'] >= '18:00:00') & (df['Global_active_power'] > 6.0)]
    
    res = res[(res['Sub_metering_2'] > res['Sub_metering_1']) & 
              (res['Sub_metering_2'] > res['Sub_metering_3'])]
    
    half_len = len(res) // 2
    first_half = res.iloc[:half_len]
    second_half = res.iloc[half_len:]
    
    return pd.concat([first_half.iloc[::3], second_half.iloc[::4]])

complex_df = complex_evening_filter(df_power)

time_taken = timeit.timeit(lambda: complex_evening_filter(df_power), number=10) / 10

print(f"Знайдено записів після фільтрації та зрізів: {len(complex_df)}")
print(f"Середній час виконання: {time_taken:.5f} секунд")
complex_df.head()

Знайдено записів після фільтрації та зрізів: 310
Середній час виконання: 0.23170 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


### Завдання 6. Статистичні перетворення та One Hot Encoding
Пронормувати та стандартизувати вибраний датасет. Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів. Провести One Hot Encoding категоріального атрибута.

In [6]:
col = 'Global_active_power'
normalized = (df_power[col] - df_power[col].min()) / (df_power[col].max() - df_power[col].min())
standardized = (df_power[col] - df_power[col].mean()) / df_power[col].std()

print("Нормування та Стандартизація")
print(f"Оригінал (перші 3):\n{df_power[col].head(3).values}")
print(f"Нормовано:\n{normalized.head(3).values}")
print(f"Стандартизовано:\n{standardized.head(3).values}\n")

pearson_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='pearson')
spearman_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='spearman')

print("Кореляція між Потужністю та Силою струму")
print(f"Пірсон: {pearson_corr:.4f}")
print(f"Спірмен: {spearman_corr:.4f}\n")

print("One Hot Encoding")
print("Створюємо ознаку 'Month' та кодуємо її...")
df_power['Month'] = df_power['Date'].str.split('/').str[1]

df_ohe = pd.get_dummies(df_power, columns=['Month'])

print(f"Було колонок: {df_power.shape[1]}")
print(f"Стало колонок після OHE: {df_ohe.shape[1]}")
df_ohe[[col for col in df_ohe.columns if 'Month_' in col]].head()

Нормування та Стандартизація
Оригінал (перші 3):
[4.216 5.36  5.374]
Нормовано:
[0.37479631 0.47836321 0.47963064]
Стандартизовано:
[2.95507634 4.03708364 4.05032499]

Кореляція між Потужністю та Силою струму
Пірсон: 0.9989
Спірмен: 0.9954

One Hot Encoding
Створюємо ознаку 'Month' та кодуємо її...
Було колонок: 10
Стало колонок після OHE: 21


,Month_1,Month_10,Month_11,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
0,False,False,False,True,False,False,False,False,False,False,False,False
1,False,False,False,True,False,False,False,False,False,False,False,False
2,False,False,False,True,False,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,False,False
4,False,False,False,True,False,False,False,False,False,False,False,False
